# Lab 3: Decision Making and Actuation Control Logic

Welcome to Lab 3. In this lab, we will build the final control layer of an ADAS pipeline: transforming perception into executable motor commands.

---

## Overview

After the system has:
- Detected objects  
- Estimated distances  
- Filtered relevant targets (ROI)  

The final step is **decision making and control**

This lab consists of three main components:

1. Emergency Braking System (EBS)  
2. Adaptive Cruise Control (PID)  
3. Control Optimization (Deadband + Slew Rate)  

---

## Learning Objectives

- Understand safety-first control logic  
- Implement a PID controller  
- Reduce noise using deadband  
- Smooth actuator signals using slew rate limiting  

---

In [ ]:
import numpy as np

# 1. Emergency Braking System (EBS)

## Theory

If the distance to an obstacle is below a critical threshold,  
the system must stop immediately.

This is a **high-priority interrupt**:
- It bypasses all other control logic  
- It guarantees collision avoidance  

In [ ]:
CMD_STOP = 8

def emergency_brake(distance, threshold=10.0):
    """
    distance: measured distance (cm)
    threshold: safety threshold
    """

    ### TODO ###
    if distance <= threshold:
        return CMD_STOP
    
    return None

# 2. Adaptive Cruise Control (PID)

## Theory

Error:
e(t) = D_measured - D_target

Control law:
u(t) = Kp * e + Ki * integral + Kd * derivative

In [1]:
class PIDController:
    def __init__(self, Kp, Ki, Kd):
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        
        self.integral = 0
        self.prev_error = 0

    def update(self, distance, target, dt=1.0):
        ### TODO ###
        error = distance - target
        
        # tích phân
        self.integral += error * dt
        
        # đạo hàm
        derivative = (error - self.prev_error) / dt[...]
        
        # PID output
        output = (
            self.Kp * error +
            self.Ki * self.integral +
            self.Kd * derivative
        )
        self.prev_error = error
        return output

# 3. Deadband

## Theory

If the error is small (|e| < threshold),  
we treat it as zero.

This prevents unnecessary micro-adjustments.

In [ ]:
def apply_deadband(error, threshold=2.5):
    ### TODO ###
    if abs(error) < threshold:
        return 0.0
    return error

# 4. Slew Rate Limiting

## Theory

Limit how fast the control output can change:

delta = output - prev_output

If delta exceeds a threshold → clamp it

In [ ]:
def apply_slew_rate(output, prev_output, max_delta=5.0):
    delta = output - prev_output

    ### TODO ###
    if delta > max_delta:
        output = prev_output + max_delta
    elif delta < -max_delta:
        output = prev_output - max_delta

    return output

# 5. Integrated Control Pipeline

Processing order:

1. Check Emergency Brake  
2. If safe:
   - Compute PID  
   - Apply Deadband  
   - Apply Slew Rate  

In [ ]:
def control_step(distance, target, pid, prev_output):

    # 1. Emergency Brake
    stop_cmd = emergency_brake(distance)
    if stop_cmd is not None:
        return stop_cmd, 0

    # 2. PID
    raw_output = pid.update(distance, target)

    # TODO: compute error
    error = distance - target

    # TODO: apply deadband
    error = apply_deadband(error)

    # TODO: recompute or adjust output if needed
    output = (
        pid.Kp * error +
        pid.Ki * pid.integral +
        pid.Kd * derivative
    )
    # 3. Slew rate
    output = apply_slew_rate(output, prev_output)

    return 1, output  # 1 = normal driving

In [ ]:
pid = PIDController(Kp=2.0, Ki=0.1, Kd=1.0)

distances = [20, 18, 15, 13, 12, 11, 9]
prev_output = 0

print("Simulation:")

for d in distances:
    cmd, output = control_step(d, target=12.5, pid=pid, prev_output=prev_output)
    
    print(f"Distance: {d} | CMD: {cmd} | Output: {output:.2f}")
    
    prev_output = output